# Week 3 Lab: Building a Simple Agent
**Research Engineer:** Victor Okoye

**Goal:** Build a text loop that forces the AI to think, ask for a tool, and read the result before answering. We are running this locally so no API keys are needed.

In [1]:
# Install required libraries
!pip install -q transformers torch accelerate

import torch
from transformers import pipeline

# Load the model directly into Kaggle's GPU
print("Loading local model...")
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
)

Loading local model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 1. The Tool (Our Fake Database)
The AI cannot browse the internet. We must give it a Python function to act as its source of truth. If it asks for Lagos, it gets 15 million.

In [2]:
def get_population(city):
    if "Lagos" in city:
        return "15 million people"
    elif "Abuja" in city:
        return "3 million people"
    else:
        return "I do not know the population of that city."

## 2. The Engine (The Python Loop)
This is the core of the agent. It takes instructions and a question. It gives the AI 3 chances to answer. If the AI types `Action:`, Python intercepts it, runs our tool, and feeds the observation back to the AI.

In [3]:
def run_simple_agent(instructions, question):
    print("User Question:", question)
    print("-------------------------")
    
    # Start the chat history with our instructions
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]
    
    # Give the AI up to 3 turns
    for step in range(3):
        # Ask the AI to generate text
        prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        outputs = pipe(prompt, max_new_tokens=100, do_sample=False, temperature=0.1)
        
        ai_text = outputs[0]["generated_text"][len(prompt):].strip()
        print("\nAI Wrote:")
        print(ai_text)
        
        # Stop if it has the final answer
        if "Final Answer:" in ai_text:
            print("\nThe agent finished the task.")
            break
            
        # Check if it is asking for the tool
        if "Action:" in ai_text:
            try:
                # Extract the city name
                city_name = ai_text.split("Input:")[1].strip()
                
                # Run the Python function
                observation = get_population(city_name)
                
                print("\n[Python Script Found]:", observation)
                print("-------------------------")
                
                # Save the progress so the AI remembers it on the next loop
                messages.append({"role": "assistant", "content": ai_text})
                messages.append({"role": "user", "content": "Observation: " + observation})
                
            except Exception:
                print("\n[System Error]: AI format was wrong.")
                messages.append({"role": "assistant", "content": ai_text})
                messages.append({"role": "user", "content": "Observation: Format error. Use Input: [city]"})

## 3. The Failure Case (Zero-Shot)
Small models like Qwen 0.5B are eager to please. If we just give them rules without examples, they often ignore the rules, hallucinate an answer, and completely bypass our tool. Let us test a basic instruction set.

In [4]:
bad_instructions = """You are an agent. You can look up the population of cities.
If you do not know the answer, use this format:
Thought: I need to find the population.
Action: lookup
Input: [city name]

When you have the answer, write:
Final Answer: [your answer]
"""

# Watch the model fail to use the tool and hallucinate "24 million" instead
run_simple_agent(bad_instructions, "Can you tell me the population of Lagos?")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User Question: Can you tell me the population of Lagos?
-------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



AI Wrote:
Thought: To find the population of Lagos, I should first determine which city it is located in and then check its official population data.
Action: lookup
Input: Lagos

[Python Script Found]: 15 million people
-------------------------

AI Wrote:
Thought: The population of Lagos is 15 million people.
Final Answer: The population of Lagos is 15 million people.

The agent finished the task.


## 4. The Fix (One-Shot Example)
To fix this, we combine Week 2 and Week 3. We must show the model exactly what a successful loop looks like. By providing a strict example in the instructions, the Transformer's attention mechanism copies the pattern.

In [5]:
good_instructions = """You are a strict ReAct AI agent. You MUST use the lookup tool to answer questions. NEVER guess from memory.

Use EXACTLY this format:
Thought: I need to find the population.
Action: lookup
Input: [city name]

Wait for the Observation. 

When you have the answer, write it like this:
Final Answer: [your answer]

EXAMPLE:
User: Can you tell me the population of Abuja?
Thought: I need to find the population of Abuja using my tool.
Action: lookup
Input: Abuja
"""

# Now the model will stop, ask for the tool, and wait for Python to give it the 15 million figure.
run_simple_agent(good_instructions, "Can you tell me the population of Lagos?")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User Question: Can you tell me the population of Lagos?
-------------------------


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



AI Wrote:
Thought: I need to find the population of Lagos using my tool.
Action: lookup
Input: Lagos

[Python Script Found]: 15 million people
-------------------------

AI Wrote:
Final Answer: The population of Lagos is approximately 15 million people.

The agent finished the task.
